# At-Least-Once vs. Exactly-Once Semantics

When a streaming pipeline crashes mid-process, the way your data is handled depends on the processing guarantee:

---

## At-Least-Once

- **Guarantee:** No data is lost. Every message is processed at least once.
- **Caveat:** Some records may be processed and written twice, resulting in duplicate rows if a failure occurs before Spark records its progress.
- **Example Scenario:** If the cluster crashes after writing a batch to storage but before saving the read position, Spark will re-read and re-write that batch when restarted.

---

## Exactly-Once

- **Guarantee:** Each record is processed exactly one time—no missing data, no duplicates.
- **How Spark + Delta Lake Achieves This:**
  1. **Replayable Source:** The stream source (like Azure Event Hubs or Kafka) allows Spark to rewind to a specific message position (offset).
  2. **Checkpointing:** Spark saves stream progress and metadata to cloud storage.
  3. **Idempotent Sink:** Delta Lake uses a transaction log. If Spark tries to write the same micro-batch twice, Delta recognizes the duplicate and skips the extra commit.

---

# Checkpointing and Fault Tolerance

## Checkpointing

Checkpointing acts as a permanent bookmark on durable cloud storage (such as ADLS or Unity Catalog Volumes). After each successful micro-batch, Spark writes two tracking records to the checkpoint directory:

- **offsets:** Tracks how far Spark has read from the queue (e.g., "read up to offset 4500").
- **commits:** Confirms that the micro-batch was successfully committed to the target table.

---

## Fault Tolerance

Checkpoints make the pipeline resilient:

1. If a cluster crashes or terminates after finishing a job with `trigger(availableNow=True)`, a new cluster can start up.
2. Spark checks the checkpoint directory.
3. It reads the last committed offset and resumes processing from where it stopped, without scanning the entire dataset again.